In [2]:
import pandas as pd

df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

print(df.shape)
print(df.dtypes)

(1470, 35)
Age                         int64
Attrition                     str
BusinessTravel                str
DailyRate                   int64
Department                    str
DistanceFromHome            int64
Education                   int64
EducationField                str
EmployeeCount               int64
EmployeeNumber              int64
EnvironmentSatisfaction     int64
Gender                        str
HourlyRate                  int64
JobInvolvement              int64
JobLevel                    int64
JobRole                       str
JobSatisfaction             int64
MaritalStatus                 str
MonthlyIncome               int64
MonthlyRate                 int64
NumCompaniesWorked          int64
Over18                        str
OverTime                      str
PercentSalaryHike           int64
PerformanceRating           int64
RelationshipSatisfaction    int64
StandardHours               int64
StockOptionLevel            int64
TotalWorkingYears           int64
Tra

In [3]:
# Null check
print(df.isnull().sum())

print("---")

# Confirm constant columns
print(df['EmployeeCount'].unique())
print(df['Over18'].unique())
print(df['StandardHours'].unique())

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

### AUDIT FINDING 1: Missing Values
No missing values across all 35 columns.
No imputation required.

### AUDIT FINDING 2: Constant Columns
Three constant columns identified with a single value across all 1,470 rows: - zero analytical value.
- EmployeeCount: all values = 1
- Over18: all values = 'Y'
- StandardHours: all values = 80

 Decision: drop all three before modelling and schema design.

In [4]:
print(df['Attrition'].value_counts())
print("---")
print(df['Attrition'].value_counts(normalize=True).round(3) * 100)

Attrition
No     1233
Yes     237
Name: count, dtype: int64
---
Attrition
No     83.9
Yes    16.1
Name: proportion, dtype: float64


### Task 3: Target Variable Distribution (Attrition)

Checked the distribution of the target variable to assess class balance.

### Findings
- No (Stayed): 1,233 employees - 83.9%
- Yes (Left): 237 employees - 16.1%

The dataset is imbalanced at roughly a 5:1 ratio. The 16.1% attrition rate aligns 
with the figure stated in the Vertex Pharma engagement brief.

### Why This Matters
A classifier trained on imbalanced data can achieve 83.9% accuracy by predicting 
"No" for every employee without learning anything meaningful. This makes accuracy 
a misleading metric for this problem.

### Decisions
- Apply SMOTE (Synthetic Minority Oversampling Technique) during model training 
  to balance the classes in the training set.
- Evaluate model performance using precision, recall, F1-score, and ROC-AUC 
  rather than accuracy.

In [5]:
# Cardinality check on all categorical columns
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts())
    print("---")

Attrition: 2 unique values
Attrition
No     1233
Yes     237
Name: count, dtype: int64
---
BusinessTravel: 3 unique values
BusinessTravel
Travel_Rarely        1043
Travel_Frequently     277
Non-Travel            150
Name: count, dtype: int64
---
Department: 3 unique values
Department
Research & Development    961
Sales                     446
Human Resources            63
Name: count, dtype: int64
---
EducationField: 6 unique values
EducationField
Life Sciences       606
Medical             464
Marketing           159
Technical Degree    132
Other                82
Human Resources      27
Name: count, dtype: int64
---
Gender: 2 unique values
Gender
Male      882
Female    588
Name: count, dtype: int64
---
JobRole: 9 unique values
JobRole
Sales Executive              326
Research Scientist           292
Laboratory Technician        259
Manufacturing Director       145
Healthcare Representative    131
Manager                      102
Sales Representative          83
Research Director    

C:\Users\hp\AppData\Local\Temp\ipykernel_19208\2925364793.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


In [6]:
print(df.describe())


               Age    DailyRate  DistanceFromHome    Education  EmployeeCount  \
count  1470.000000  1470.000000       1470.000000  1470.000000         1470.0   
mean     36.923810   802.485714          9.192517     2.912925            1.0   
std       9.135373   403.509100          8.106864     1.024165            0.0   
min      18.000000   102.000000          1.000000     1.000000            1.0   
25%      30.000000   465.000000          2.000000     2.000000            1.0   
50%      36.000000   802.000000          7.000000     3.000000            1.0   
75%      43.000000  1157.000000         14.000000     4.000000            1.0   
max      60.000000  1499.000000         29.000000     5.000000            1.0   

       EmployeeNumber  EnvironmentSatisfaction   HourlyRate  JobInvolvement  \
count     1470.000000              1470.000000  1470.000000     1470.000000   
mean      1024.865306                 2.721769    65.891156        2.729932   
std        602.024335            

### Task 5: Numerical Column Distributions

Ran df.describe() across all 26 numeric columns to check for anomalies, 
impossible values, and additional constants.

### Findings

**Age:** Range 18 to 60, mean 36.9. Plausible working population distribution.

**TotalWorkingYears:** Min of 0 (first year employees) to max of 40, mean 11.3. 
Realistic range, no anomalies.

**YearsSinceLastPromotion:** Median of 1 year but max of 15, std of 3.2. 
Long tail of employees without recent promotion. Flagged as a likely strong 
attrition predictor.

**YearsAtCompany:** Median 5 years, mean 7, max 40. Workforce skews relatively 
junior in tenure, typical for pharma sales and lab roles.

**EmployeeNumber:** ID field only (1 to 2068). No analytical value. 
Will be used as primary key in the fact table.

**StandardHours:** Confirmed constant — all values = 80, std = 0.0. 
Dropping alongside EmployeeCount and Over18.

### Overall Assessment
All numeric columns fall within plausible real-world ranges. No corrupt values, 
no impossible entries, no unexpected distributions requiring transformation 
at this stage.

### Decisions
- Drop EmployeeCount, Over18, and StandardHours (confirmed constants, zero variance).
- Retain YearsSinceLastPromotion as a priority feature for the predictive model.
- MonthlyIncome expected to be right-skewed (typical for salary data) -
  will apply log transformation before modelling if required.
- EmployeeNumber to serve as primary key in the fact table only.

In [9]:
# Reload and resave with clean UTF-8 encoding, no BOM
df.to_csv('WA_Clean.csv', index=False, encoding='utf-8')

In [10]:
print(df.columns.tolist())

['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [11]:
import pandas as pd

df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Drop constant columns
df.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'], inplace=True)

print(df.shape)

(1470, 32)


In [13]:
# 1. Promotion gap: how long since promotion relative to time at company
df['PromotionGap'] = df['YearsAtCompany'] - df['YearsSinceLastPromotion']

# 2. Tenure ratio: proportion of total career spent at Vertex
df['TenureRatio'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1)

# 3. Income per job level: compensation relative to seniority
df['IncomePerJobLevel'] = df['MonthlyIncome'] / df['JobLevel']

# 4. Satisfaction index: average of all four satisfaction scores
df['SatisfactionIndex'] = (
    df['EnvironmentSatisfaction'] +
    df['JobSatisfaction'] +
    df['RelationshipSatisfaction'] +
    df['WorkLifeBalance']
) / 4

print(df[['PromotionGap', 'TenureRatio', 'IncomePerJobLevel', 'SatisfactionIndex']].head())

   PromotionGap  TenureRatio  IncomePerJobLevel  SatisfactionIndex
0             6     0.666667             2996.5               2.00
1             9     0.909091             2565.0               3.00
2             0     0.000000             2090.0               3.00
3             5     0.888889             2909.0               3.25
4             0     0.285714             3468.0               2.50


In [14]:
# Encode the target variable
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Encode binary columns
df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

# One-hot encode remaining categorical columns
df = pd.get_dummies(df, columns=[
    'BusinessTravel',
    'Department',
    'EducationField',
    'JobRole',
    'MaritalStatus'
], drop_first=False)

print(df.shape)
print(df.dtypes.tail(20))

(1470, 55)
Department_Research & Development    bool
Department_Sales                     bool
EducationField_Human Resources       bool
EducationField_Life Sciences         bool
EducationField_Marketing             bool
EducationField_Medical               bool
EducationField_Other                 bool
EducationField_Technical Degree      bool
JobRole_Healthcare Representative    bool
JobRole_Human Resources              bool
JobRole_Laboratory Technician        bool
JobRole_Manager                      bool
JobRole_Manufacturing Director       bool
JobRole_Research Director            bool
JobRole_Research Scientist           bool
JobRole_Sales Executive              bool
JobRole_Sales Representative         bool
MaritalStatus_Divorced               bool
MaritalStatus_Married                bool
MaritalStatus_Single                 bool
dtype: object


In [16]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=['Attrition', 'EmployeeNumber'])
y = df['Attrition']

# Split into training and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training attrition rate: {y_train.mean().round(3)}")
print(f"Test attrition rate: {y_test.mean().round(3)}")

Training set: (1176, 53)
Test set: (294, 53)
Training attrition rate: 0.162
Test attrition rate: 0.16


In [17]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE: {y_train_sm.value_counts().to_dict()}")

Before SMOTE: {0: 986, 1: 190}
After SMOTE: {0: 986, 1: 986}


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

# Train baseline model
lr = LogisticRegression(max_iter=2000, random_state=42)
lr.fit(X_train_scaled, y_train_sm)

# Evaluate on test set
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results")
print("---")
print(classification_report(y_test, y_pred_lr, target_names=['Stayed', 'Left']))
print(f"ROC-AUC Score: {round(roc_auc_score(y_test, y_prob_lr), 3)}")

Logistic Regression Results
---
              precision    recall  f1-score   support

      Stayed       0.89      0.96      0.93       247
        Left       0.67      0.38      0.49        47

    accuracy                           0.87       294
   macro avg       0.78      0.67      0.71       294
weighted avg       0.86      0.87      0.86       294

ROC-AUC Score: 0.812


In [20]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train_sm, y_train_sm)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest Results")
print("---")
print(classification_report(y_test, y_pred_rf, target_names=['Stayed', 'Left']))
print(f"ROC-AUC Score: {round(roc_auc_score(y_test, y_prob_rf), 3)}")

Random Forest Results
---
              precision    recall  f1-score   support

      Stayed       0.87      0.95      0.91       247
        Left       0.52      0.28      0.36        47

    accuracy                           0.84       294
   macro avg       0.70      0.61      0.64       294
weighted avg       0.82      0.84      0.82       294

ROC-AUC Score: 0.755


In [21]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    scale_pos_weight=5,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_train_sm, y_train_sm)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGBoost Results")
print("---")
print(classification_report(y_test, y_pred_xgb, target_names=['Stayed', 'Left']))
print(f"ROC-AUC Score: {round(roc_auc_score(y_test, y_prob_xgb), 3)}")

XGBoost Results
---
              precision    recall  f1-score   support

      Stayed       0.88      0.93      0.90       247
        Left       0.45      0.32      0.38        47

    accuracy                           0.83       294
   macro avg       0.67      0.62      0.64       294
weighted avg       0.81      0.83      0.82       294

ROC-AUC Score: 0.761


### Model Selection: Logistic Regression

Three models were evaluated on the test set:

| Model | ROC-AUC | Recall (Left) |
|---|---|---|
| Logistic Regression | 0.812 | 0.38 |
| XGBoost | 0.761 | 0.32 |
| Random Forest | 0.755 | 0.28 |

Logistic Regression outperformed both ensemble models on ROC-AUC 
and recall for the minority class (Left).

This is consistent with known behaviour on small, clean, tabular 
datasets where relationships between features and the target are 
relatively linear. With only 1,470 rows, ensemble methods like 
Random Forest and XGBoost did not have enough data to fully exploit 
their complexity advantage.

### Decision
Logistic Regression selected as the final model for the following reasons:

- Highest ROC-AUC score (0.812)
- Best recall on the Left class (0.38)
- Most interpretable for client-facing recommendations
- Coefficients can be directly communicated to non-technical stakeholders

In a consulting context, a model that a Board can understand and trust 
is more valuable than a marginally more complex model with similar performance.

In [23]:
import pandas as pd
import matplotlib.pyplot as plt

# Get feature importance from logistic regression coefficients
feature_names = X_train_sm.columns if hasattr(X_train_sm, 'columns') else X.columns
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).head(15)

print(coef_df.to_string(index=False))

                         Feature  Coefficient
               SatisfactionIndex     2.075124
                   MonthlyIncome     1.711599
         EnvironmentSatisfaction    -1.612308
                 JobSatisfaction    -1.547470
            MaritalStatus_Single     1.544917
        RelationshipSatisfaction    -1.402490
   JobRole_Laboratory Technician     1.378390
           MaritalStatus_Married     1.229985
    EducationField_Life Sciences     1.181146
BusinessTravel_Travel_Frequently     1.179106
                 WorkLifeBalance    -1.112702
          EducationField_Medical     1.060834
    JobRole_Sales Representative     1.051051
 EducationField_Technical Degree     1.050882
                Department_Sales     1.022692


### Modelling Note: Multicollinearity in Feature Coefficients

Two features show counterintuitive positive coefficients that contradict 
the SQL exploratory findings:

- SatisfactionIndex (coefficient: +2.075) — should reduce attrition
- MonthlyIncome (coefficient: +1.711) — should reduce attrition

This is caused by multicollinearity. SatisfactionIndex was engineered 
from EnvironmentSatisfaction, JobSatisfaction, and RelationshipSatisfaction, 
so all four features are highly correlated. Similarly, IncomePerJobLevel 
was derived from MonthlyIncome, creating correlation between the two.

When features are strongly correlated, logistic regression cannot cleanly 
separate their individual effects and coefficients become unreliable.

### Reliable coefficients (directionally consistent with SQL findings):
- EnvironmentSatisfaction: -1.612 (higher satisfaction → lower attrition)
- JobSatisfaction: -1.547 (higher satisfaction → lower attrition)
- RelationshipSatisfaction: -1.402 (higher satisfaction → lower attrition)
- WorkLifeBalance: -1.113 (better balance → lower attrition)
- MaritalStatus_Single: +1.545 (single employees → higher attrition)
- JobRole_Laboratory Technician: +1.378 (confirms SQL finding)
- JobRole_Sales Representative: +1.051 (confirms SQL finding)
- BusinessTravel_Travel_Frequently: +1.179 (frequent travel → higher attrition)

### Decision:
In a production model, either SatisfactionIndex or the four individual 
satisfaction scores would be dropped to reduce multicollinearity. 
For this engagement the individual scores are retained as they are 
more interpretable for client recommendations.

## If Deployed in Production: Feature Refinement Options

Three approaches to handling the multicollinearity in production:

**Option 1: Drop the composite index, keep individual scores (recommended)**
Remove SatisfactionIndex and retain EnvironmentSatisfaction, JobSatisfaction,
RelationshipSatisfaction, and WorkLifeBalance separately. Simplest fix and 
keeps the model interpretable for client presentations.

**Option 2: Drop individual scores, keep composite index**
Remove the four individual scores and retain only SatisfactionIndex.
Reduces four correlated features to one but loses granularity in the 
feature importance output.

**Option 3: Principal Component Analysis (PCA)**
Mathematically combine the correlated satisfaction scores into one or two 
abstract components that capture the same information without the correlation 
problem. Most technically rigorous but harder to explain to a non-technical 
client audience.

For a Big 4 consulting engagement, Option 1 is preferred. Interpretability 
matters more than marginal performance gains. Being able to tell a client 
"job satisfaction is a strong driver of attrition" in plain English is 
more valuable than a model that is 1% more accurate but nobody understands.

In [24]:
# Generate risk scores for all employees
df_scored = df.copy()
feature_cols = X.columns
X_all = df_scored[feature_cols]
X_all_scaled = scaler.transform(X_all)

df_scored['AttritionRiskScore'] = lr.predict_proba(X_all_scaled)[:, 1]
df_scored['RiskBand'] = pd.cut(
    df_scored['AttritionRiskScore'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High']
)

print(df_scored['RiskBand'].value_counts())
print(df_scored.groupby('RiskBand')['AttritionRiskScore'].mean().round(3))

RiskBand
Low       1187
Medium     182
High       101
Name: count, dtype: int64
RiskBand
Low       0.074
Medium    0.425
High      0.773
Name: AttritionRiskScore, dtype: float64


### Retention Priority Segmentation

Employees scored using the logistic regression model and segmented 
into three risk bands based on predicted attrition probability:

| Risk Band | Employee Count | Avg Risk Score | Interpretation |
|---|---|---|---|
| Low | 1,187 | 0.074 | 7.4% predicted probability of leaving. No immediate action required. |
| Medium | 182 | 0.425 | 42.5% predicted probability. Monitor and target with light-touch interventions. |
| High | 101 | 0.773 | 77.3% predicted probability. Immediate intervention recommended. |

### Why This Matters
Rather than presenting attrition as an abstract percentage, the model 
identifies 101 specific employees at high risk of leaving in the next 
6 to 12 months. This enables Vertex Pharma to prioritise retention 
spend where it will have the greatest impact rather than applying 
blanket interventions across the whole workforce.

### Next Step
Profile the 101 high risk employees by department, job role, income, 
and overtime status to identify common characteristics and inform 
targeted intervention design.

In [26]:
high_risk = df_scored[df_scored['RiskBand'] == 'High'].copy()

print(f"High Risk Employees: {len(high_risk)}")
print("---")

# Department breakdown
print("By Department:")
print(high_risk[['Department_Sales', 
                  'Department_Research & Development',
                  'Department_Human Resources']].sum())
print("---")

# Job Level
print("By Job Level:")
print(high_risk['JobLevel'].value_counts().sort_index())
print("---")

# Overtime
print("By Overtime:")
print(high_risk['OverTime'].value_counts())
print("---")

# Income distribution
print("Monthly Income Stats:")
print(high_risk['MonthlyIncome'].describe().round(0))
print("---")

# Marital Status
print("By Marital Status:")
print(high_risk['MaritalStatus_Single'].value_counts())

High Risk Employees: 101
---
By Department:
Department_Sales                     49
Department_Research & Development    47
Department_Human Resources            5
dtype: int64
---
By Job Level:
JobLevel
1    65
2    26
3     8
4     2
Name: count, dtype: int64
---
By Overtime:
OverTime
1    60
0    41
Name: count, dtype: int64
---
Monthly Income Stats:
count      101.0
mean      3996.0
std       2688.0
min       1091.0
25%       2293.0
50%       2837.0
75%       5326.0
max      13758.0
Name: MonthlyIncome, dtype: float64
---
By Marital Status:
MaritalStatus_Single
True     73
False    28
Name: count, dtype: int64


### Profile of High Risk Employees (101 Employees)

| Attribute | Finding |
|---|---|
| Department | Sales (49), R&D (47), HR (5) |
| Job Level | 65% at Level 1, 91% at Levels 1-2 |
| Overtime | 59% work overtime vs 28% organisation average |
| Median Income | £2,837/month (majority below £3,000 band) |
| Marital Status | 72% single |

### Summary Profile
The typical high risk employee at Vertex Pharma is junior (Job Level 1), 
single, working overtime, and earning below £3,000 per month in either 
Sales or Research & Development.

### Intervention Implications
- Compensation review for Job Level 1 and 2 roles in Sales and R&D
- Overtime reduction or compensation for junior employees
- Structured career progression pathways from Level 1 to Level 2
- Targeted engagement initiatives for single employees with low tenure

In [27]:
# Select relevant columns for Power BI export
cols_to_export = [
    'EmployeeNumber', 'Age', 'Gender', 'MaritalStatus_Single',
    'MaritalStatus_Married', 'OverTime', 'Attrition',
    'MonthlyIncome', 'JobLevel', 'TotalWorkingYears',
    'YearsAtCompany', 'YearsSinceLastPromotion',
    'EnvironmentSatisfaction', 'JobSatisfaction',
    'WorkLifeBalance', 'PerformanceRating',
    'SatisfactionIndex', 'IncomePerJobLevel',
    'AttritionRiskScore', 'RiskBand'
]

df_export = df_scored[cols_to_export].copy()
df_export.to_csv('VP_EmployeeRiskScores.csv', index=False)
print(f"Exported {len(df_export)} rows")
print(df_export[['EmployeeNumber', 'AttritionRiskScore', 'RiskBand']].head())

Exported 1470 rows
   EmployeeNumber  AttritionRiskScore RiskBand
0               1            0.741823     High
1               2            0.019666      Low
2               4            0.577503   Medium
3               5            0.084710      Low
4               7            0.403079   Medium


In [28]:
# Export original data with risk scores added
df_original = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
df_original['AttritionRiskScore'] = df_scored['AttritionRiskScore'].values
df_original['RiskBand'] = df_scored['RiskBand'].values
df_original.to_csv('VP_Dashboard_Data.csv', index=False)
print(f"Dashboard export: {len(df_original)} rows")

Dashboard export: 1470 rows
